# MLP Gating Analysis

Analyze MLP activation patterns as implicit gating: activation sparsity, pre/post nonlinearity interaction, neuron selectivity, cross-layer sparsity trends, and per-neuron output contribution decomposition.

## Why This Matters

MLP gating provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_gating_analysis import (
    gate_activation_patterns,
    gate_value_interaction,
    gate_selectivity,
    gating_sparsity,
    gate_contribution_decomposition,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
# Activation patterns and sparsity
result = gate_activation_patterns(model, tokens, layer=0)
print(f'Sparsity: {result["sparsity"]:.4f}')
print(f'Mean activation: {result["mean_activation"]:.4f}')
print('Top neurons (idx, mean_magnitude):')
for idx, mag in result['top_neurons'][:5]:
    print(f'  Neuron {idx}: {mag:.4f}')

In [ ]:
# Pre vs post activation (nonlinearity effect)
result = gate_value_interaction(model, tokens, layer=0)
print(f'Pre-post correlation: {result["pre_post_correlation"]:.4f}')
print(f'Direction change: {result["direction_change"]:.4f}')
print(f'Magnitude change: {result["magnitude_change"]:.4f}')
print(f'Amplified: {result["n_amplified"]}, Suppressed: {result["n_suppressed"]}')

In [ ]:
# Neuron selectivity across inputs
tokens_list = [
    jnp.array([1, 15, 30, 45, 60]),
    jnp.array([50, 65, 80, 95, 10]),
    jnp.array([20, 40, 60, 80, 99]),
]
result = gate_selectivity(model, tokens_list, layer=0)
print(f'Always on: {result["always_on"]}, Always off: {result["always_off"]}')
print('Most selective neurons:')
for idx, score in result['most_selective'][:5]:
    print(f'  Neuron {idx}: selectivity={score:.4f}')

In [ ]:
# Sparsity across all layers
result = gating_sparsity(model, tokens)
print(f'Overall sparsity: {result["overall_sparsity"]:.4f}')
print(f'Sparsity trend: {result["sparsity_trend"]:.4f}')
for layer in result['per_layer']:
    print(f'  Layer {layer["layer"]}: sparsity={layer["sparsity"]:.3f}, '
          f'dead={layer["dead_neurons"]}')

In [ ]:
# Per-neuron contribution decomposition
result = gate_contribution_decomposition(model, tokens, layer=0)
print(f'Total output norm: {result["total_output_norm"]:.4f}')
print(f'Top-k concentration: {result["concentration"]:.4f}')
print(f'Reconstruction error (top-k): {result["reconstruction_error"]:.4f}')
for c in result['top_contributors'][:5]:
    print(f'  Neuron {c["neuron"]}: norm={c["contribution_norm"]:.4f}, '
          f'align={c["alignment_with_output"]:.3f}')